<a href="https://colab.research.google.com/github/Existanze54/sirius-neural-networks-2026/blob/main/Seminars/S01_Intro_and_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Нейронные сети в биоинформатике

### Семинар 1: Знакомство с библиотекой PyTorch

In [ ]:
import torch

In [ ]:
torch.__version__

'2.2.1+cu121'

![PyTorch 2.0](https://data.bioml.ru/htdocs/courses/bioml/neural_networks/pytorch/img/pytorch_2.png)

In [ ]:
import numpy as np

Если забылось, рекомендуем вспомнить основы, почитав [лекцию](https://colab.research.google.com/drive/1UZPtzIyxCTlKjyeyrrHj2zt8Z3qwWEfc) про NumPy.

**PyTorch** - одна из самых популярных на сегодня бибиблиотек для написания и обучения нейронных сетей.

Есть альтернатива: ~утки~ TensorFlow + Keras, но мы их не будем рассматривать в этом курсе.

## Тензоры и векторизация вычислений

Как и в NumPy, в PyTorch имплементирован класс многомерного массива данных (не путать с тензорами в физике, называется он здесь `Tensor`, но на самом деле это массив с дополнительным функционалом).

Для любопытных: раньше было 2 класса: `Tensor` и `Variable`, но в конце-концов `Variable` упразднили и остался только `Tensor`...


Создадим тензор! Сделать это можно несколькими способами:

In [ ]:
x = torch.tensor([[0.2, 0.5, 0.8],
                  [1.3, 1.7, 8.0]])
x

tensor([[0.2000, 0.5000, 0.8000],
        [1.3000, 1.7000, 8.0000]])

Можно сгенерировать псевдослучайные данные:

In [ ]:
x = torch.rand(size=(2, 3))
x

tensor([[0.5421, 0.0563, 0.8661],
        [0.4258, 0.8212, 0.2117]])

Можно просто аллоцировать некие произвольные данные из памяти:

In [ ]:
torch.empty((2, 3))

tensor([[2.6048e+05, 4.5576e-41, 2.6048e+05],
        [4.5576e-41, 7.4937e+31, 1.7753e+28]])

Или можно загрузить данные из NumPy:

In [ ]:
x = np.random.normal(size=(2, 3)).astype(np.float32)
y = torch.from_numpy(x)

In [ ]:
y

tensor([[-0.0243, -0.9843, -1.5323],
        [-1.5652,  0.4349,  0.0736]])

Обратите внимание, что `x` и `y` расположены в одном блоке памяти:

In [ ]:
y[1, 1] = -10
x

array([[ -0.02431436,  -0.9843401 ,  -1.5322767 ],
       [ -1.5652288 , -10.        ,   0.07359219]], dtype=float32)

In [ ]:
x[1, 1] = 5
y

tensor([[-0.0243, -0.9843, -1.5323],
        [-1.5652,  5.0000,  0.0736]])

### Свойства тензора

In [ ]:
dir(x)

['H',
 'T',
 '__abs__',
 '__add__',
 '__and__',
 '__array__',
 '__array_priority__',
 '__array_wrap__',
 '__bool__',
 '__class__',
 '__complex__',
 '__contains__',
 '__deepcopy__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__div__',
 '__dlpack__',
 '__dlpack_device__',
 '__doc__',
 '__eq__',
 '__float__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__iand__',
 '__idiv__',
 '__ifloordiv__',
 '__ilshift__',
 '__imod__',
 '__imul__',
 '__index__',
 '__init__',
 '__init_subclass__',
 '__int__',
 '__invert__',
 '__ior__',
 '__ipow__',
 '__irshift__',
 '__isub__',
 '__iter__',
 '__itruediv__',
 '__ixor__',
 '__le__',
 '__len__',
 '__long__',
 '__lshift__',
 '__lt__',
 '__matmul__',
 '__mod__',
 '__module__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '__nonzero__',
 '__or__',
 '__pos__',
 '__pow__',
 '__radd__',
 '__rand__',
 '__rdiv__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__reversed_

In [ ]:
y.dtype

torch.float32

In [ ]:
y.shape

torch.Size([2, 3])

In [ ]:
y.ndim

2

In [ ]:
y.device

device(type='cpu')

### Вычисления на GPU

_Переключиться на GPU в Google Colab можно в меню Runtime > Change runtime type_.

PyTorch позволяет использовать в вычислениях видеокарту. За счет высокой степени параллелизации вычислений и архитектурных оптимизаций для расчета на числах с плавающей запятой видеокарты позволяют ускорить вычисления в несколько раз!

In [ ]:
import torch

In [ ]:
a = torch.normal(0, 1, size=(888, 1000))
b = torch.normal(0, 2, size=(1000, 777))

In [ ]:
%%timeit
a @ b

12.7 ms ± 2.13 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


Перенесем наши данные на GPU с помощью метода `.to`

In [ ]:
device = "cuda:0"

In [ ]:
a_gpu = a.to(device)
b_gpu = b.to(device)

In [ ]:
%%timeit
a_gpu @ b_gpu

383 µs ± 2.92 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


Проверить, что у вас есть CUDA-совместимое устройство и оно распознано, можно с помощью функции `torch.cuda.is_available()`

In [ ]:
torch.cuda.is_available()

True

Иногда пишут так:

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

Перенести можно несколькими способами:

In [ ]:
a.to("cuda:0")

tensor([[-0.2216, -1.1744,  0.1709,  ..., -1.6678, -0.8869,  0.1005],
        [ 0.2014, -0.3160,  0.4397,  ..., -0.9111, -0.8044, -1.1002],
        [ 0.7851, -0.7497, -1.3362,  ...,  0.7248, -2.2241,  0.8937],
        ...,
        [ 0.5563, -0.0858,  0.8321,  ...,  0.8316, -0.3497, -0.2523],
        [-0.2741,  0.1554,  0.7055,  ..., -1.8327,  1.0443, -1.1815],
        [ 0.5862, -0.4598,  0.2673,  ..., -0.9171,  0.0588,  0.2697]],
       device='cuda:0')

In [ ]:
a.cuda()  # cuda доступен не всегда, так делать не рекомендуется

tensor([[-0.2216, -1.1744,  0.1709,  ..., -1.6678, -0.8869,  0.1005],
        [ 0.2014, -0.3160,  0.4397,  ..., -0.9111, -0.8044, -1.1002],
        [ 0.7851, -0.7497, -1.3362,  ...,  0.7248, -2.2241,  0.8937],
        ...,
        [ 0.5563, -0.0858,  0.8321,  ...,  0.8316, -0.3497, -0.2523],
        [-0.2741,  0.1554,  0.7055,  ..., -1.8327,  1.0443, -1.1815],
        [ 0.5862, -0.4598,  0.2673,  ..., -0.9171,  0.0588,  0.2697]],
       device='cuda:0')

In [ ]:
a_gpu.cpu()  # а cpu всегда доступен, так можно делать

tensor([[-0.2216, -1.1744,  0.1709,  ..., -1.6678, -0.8869,  0.1005],
        [ 0.2014, -0.3160,  0.4397,  ..., -0.9111, -0.8044, -1.1002],
        [ 0.7851, -0.7497, -1.3362,  ...,  0.7248, -2.2241,  0.8937],
        ...,
        [ 0.5563, -0.0858,  0.8321,  ...,  0.8316, -0.3497, -0.2523],
        [-0.2741,  0.1554,  0.7055,  ..., -1.8327,  1.0443, -1.1815],
        [ 0.5862, -0.4598,  0.2673,  ..., -0.9171,  0.0588,  0.2697]])

Если ошибиться и осуществлять операции с тензорами в разных источниках, получится ошибка:

In [ ]:
a @ b_gpu

RuntimeError: Expected all tensors to be on the same device, but got mat2 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_mm)

In [ ]:
a_gpu @ b

RuntimeError: Expected all tensors to be on the same device, but got mat2 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_mm)

Помните, что ошибка не всегда может быть такой красивой!

### Базовые операции с тензорами

#### Преобразования типов

In [ ]:
x = torch.tensor([1, 2, 3])
x.dtype

torch.int64

In [ ]:
y = torch.tensor([0.4, 1.5, 2.6])
y.dtype

torch.float32

In [ ]:
xf = x.float()
xf.dtype

torch.float32

In [ ]:
yf = y.int()
yf.dtype

torch.int32

In [ ]:
yf

tensor([0, 1, 2], dtype=torch.int32)

#### Бинарные операции

In [ ]:
x = torch.tensor([[0, 1, 2],
                  [3, 4, 5]])
x

tensor([[0, 1, 2],
        [3, 4, 5]])

In [ ]:
y = torch.tensor([[20, 10,  5],
                  [40, 30, 25]])
y

tensor([[20, 10,  5],
        [40, 30, 25]])

In [ ]:
z = torch.tensor([[3],
                  [7]])
z

tensor([[3],
        [7]])

In [ ]:
x + y

tensor([[20, 11,  7],
        [43, 34, 30]])

In [ ]:
x + z

tensor([[ 3,  4,  5],
        [10, 11, 12]])

#### Элементарные функции

In [ ]:
x

tensor([[0, 1, 2],
        [3, 4, 5]])

In [ ]:
x.log()

tensor([[  -inf, 0.0000, 0.6931],
        [1.0986, 1.3863, 1.6094]])

In [ ]:
x.log1p()

tensor([[0.0000, 0.6931, 1.0986],
        [1.3863, 1.6094, 1.7918]])

In [ ]:
x.exp()

tensor([[  1.0000,   2.7183,   7.3891],
        [ 20.0855,  54.5981, 148.4132]])

In [ ]:
x.sin()

tensor([[ 0.0000,  0.8415,  0.9093],
        [ 0.1411, -0.7568, -0.9589]])

In [ ]:
x.cos()

tensor([[ 1.0000,  0.5403, -0.4161],
        [-0.9900, -0.6536,  0.2837]])

In [ ]:
x.log1p().exp().log1p().exp()

tensor([[2.0000, 3.0000, 4.0000],
        [5.0000, 6.0000, 7.0000]])

#### Агрегирующие операции

In [ ]:
x

tensor([[0, 1, 2],
        [3, 4, 5]])

In [ ]:
x.mean()  # type conversion must be explicit (sometimes)!

RuntimeError: ignored

In [ ]:
x.float().mean()  # why is it Tensor?

tensor(2.5000)

In [ ]:
x = x.float()

In [ ]:
x.mean(axis=1)

tensor([1., 4.])

In [ ]:
x.max()

tensor(5.)

In [ ]:
x.argmax(axis=None)

tensor(5)

In [ ]:
x.argmax(axis=0)

tensor([1, 1, 1])

### Манипуляции формой массива

#### Индексация и фильтрация

In [ ]:
x = torch.normal(1, 1.0, size=(6, 4))
x

tensor([[-0.4974,  1.1712,  1.7070,  2.2499],
        [ 1.2007,  1.3238, -0.9118,  0.3934],
        [ 0.4878,  3.1433,  0.2791,  0.1968],
        [ 1.6048,  2.7025,  2.2585,  1.2531],
        [ 1.9028,  1.8974,  1.4985,  2.2564],
        [ 1.1477,  1.9754,  2.5848,  2.3228]])

In [ ]:
r = x[1,:]
r

tensor([ 1.2007,  1.3238, -0.9118,  0.3934])

In [ ]:
x[1,1]

tensor(1.3238)

In [ ]:
c = x[:,1]
c

tensor([1.1712, 1.3238, 3.1433, 2.7025, 1.8974, 1.9754])

In [ ]:
x_m = x.mean(axis=0)
x_m

tensor([0.9744, 2.0356, 1.2360, 1.4454])

In [ ]:
x_m > 1.4

tensor([False,  True, False,  True])

In [ ]:
r[x_m > 1.4]

tensor([1.3238, 0.3934])

In [ ]:
x[x_m > 1.4]  # не сработает

IndexError: ignored

In [ ]:
x[:,(x_m > 1.4)]

tensor([[1.1712, 2.2499],
        [1.3238, 0.3934],
        [3.1433, 0.1968],
        [2.7025, 1.2531],
        [1.8974, 2.2564],
        [1.9754, 2.3228]])

In [ ]:
x[1:3,0:3]

tensor([[ 1.2007,  1.3238, -0.9118],
        [ 0.4878,  3.1433,  0.2791]])

In [ ]:
x[[1, 2],[0, 2]]

tensor([1.2007, 0.2791])

#### Изменение формы

In [ ]:
x

tensor([[-0.4974,  1.1712,  1.7070,  2.2499],
        [ 1.2007,  1.3238, -0.9118,  0.3934],
        [ 0.4878,  3.1433,  0.2791,  0.1968],
        [ 1.6048,  2.7025,  2.2585,  1.2531],
        [ 1.9028,  1.8974,  1.4985,  2.2564],
        [ 1.1477,  1.9754,  2.5848,  2.3228]])

In [ ]:
x.shape

torch.Size([6, 4])

In [ ]:
x.reshape(shape=(8, 3))

tensor([[-0.4974,  1.1712,  1.7070],
        [ 2.2499,  1.2007,  1.3238],
        [-0.9118,  0.3934,  0.4878],
        [ 3.1433,  0.2791,  0.1968],
        [ 1.6048,  2.7025,  2.2585],
        [ 1.2531,  1.9028,  1.8974],
        [ 1.4985,  2.2564,  1.1477],
        [ 1.9754,  2.5848,  2.3228]])

In [ ]:
x = torch.arange(0.0, 24.0).reshape(shape=(6, 4))
x

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.],
        [16., 17., 18., 19.],
        [20., 21., 22., 23.]])

In [ ]:
x.reshape(shape=(4, 6))

tensor([[ 0.,  1.,  2.,  3.,  4.,  5.],
        [ 6.,  7.,  8.,  9., 10., 11.],
        [12., 13., 14., 15., 16., 17.],
        [18., 19., 20., 21., 22., 23.]])

In [ ]:
x.reshape(shape=24)  # не сработает

TypeError: ignored

In [ ]:
x.reshape(shape=(24))  # подали кортеж ведь, что не так?!

TypeError: ignored

In [ ]:
x.reshape(shape=(24,))

tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
        14., 15., 16., 17., 18., 19., 20., 21., 22., 23.])

In [ ]:
x.T

tensor([[ 0.,  4.,  8., 12., 16., 20.],
        [ 1.,  5.,  9., 13., 17., 21.],
        [ 2.,  6., 10., 14., 18., 22.],
        [ 3.,  7., 11., 15., 19., 23.]])

In [ ]:
x.permute(1, 0)

tensor([[ 0.,  4.,  8., 12., 16., 20.],
        [ 1.,  5.,  9., 13., 17., 21.],
        [ 2.,  6., 10., 14., 18., 22.],
        [ 3.,  7., 11., 15., 19., 23.]])

In [ ]:
x = torch.arange(0.0, 30.0).reshape(shape=(5, 3, 2))
x

tensor([[[ 0.,  1.],
         [ 2.,  3.],
         [ 4.,  5.]],

        [[ 6.,  7.],
         [ 8.,  9.],
         [10., 11.]],

        [[12., 13.],
         [14., 15.],
         [16., 17.]],

        [[18., 19.],
         [20., 21.],
         [22., 23.]],

        [[24., 25.],
         [26., 27.],
         [28., 29.]]])

In [ ]:
x.permute(0, 2, 1)

tensor([[[ 0.,  2.,  4.],
         [ 1.,  3.,  5.]],

        [[ 6.,  8., 10.],
         [ 7.,  9., 11.]],

        [[12., 14., 16.],
         [13., 15., 17.]],

        [[18., 20., 22.],
         [19., 21., 23.]],

        [[24., 26., 28.],
         [25., 27., 29.]]])

#### Broadcasting

In [ ]:
a = torch.arange(6.).reshape((2, 3))
a

tensor([[0., 1., 2.],
        [3., 4., 5.]])

In [ ]:
b = torch.arange(2.)
b

tensor([0., 1.])

In [ ]:
a + b

RuntimeError: ignored

In [ ]:
b.reshape((1, 2))

tensor([[0., 1.]])

In [ ]:
a + b.reshape((1, 2))

RuntimeError: ignored

In [ ]:
b.reshape((2, 1))

tensor([[0.],
        [1.]])

In [ ]:
a + b.reshape((2, 1))

tensor([[0., 1., 2.],
        [4., 5., 6.]])

In [ ]:
a[:,None,:]

tensor([[[0., 1., 2.]],

        [[3., 4., 5.]]])

Существует **правило приведения размерностей**:

1. Предположим, что `a.shape = (a_1, a_2, ..., a_n)` и `b.shape = (b_1, b_2, ..., b_n)`. Над `a` и `b` можно произвести поэлементую бинарную операцию, если $\forall \; i \in (1...n)$ выполнено хотя бы одно из условий:
    * `a_i == b_i`;
    * `a_i == 1`;
    * `b_i == 1`.


2. Если размерности не совпадают, то к массиву меньшей размерности добавляются **_ведущие_ фиктивные размерности**.

Документация:
* https://pytorch.org/docs/stable/notes/broadcasting.html
* https://docs.scipy.org/doc/numpy/user/basics.broadcasting.html (аналогично в NumPy)


In [ ]:
lonng = torch.arange(20)[None,:]  # -> (1, 20)
lonng

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19])

In [ ]:
lonng_multiplied = lonng.broadcast_to(10, 20)
lonng_multiplied

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19]])

In [ ]:
lonng = torch.arange(20)[:,None]  # -> (20, 1)
lonng

tensor([[ 0],
        [ 1],
        [ 2],
        [ 3],
        [ 4],
        [ 5],
        [ 6],
        [ 7],
        [ 8],
        [ 9],
        [10],
        [11],
        [12],
        [13],
        [14],
        [15],
        [16],
        [17],
        [18],
        [19]])

In [ ]:
lonng = torch.arange(20)[:,None]
lonng.broadcast_to(20, 10)

tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
        [ 2,  2,  2,  2,  2,  2,  2,  2,  2,  2],
        [ 3,  3,  3,  3,  3,  3,  3,  3,  3,  3],
        [ 4,  4,  4,  4,  4,  4,  4,  4,  4,  4],
        [ 5,  5,  5,  5,  5,  5,  5,  5,  5,  5],
        [ 6,  6,  6,  6,  6,  6,  6,  6,  6,  6],
        [ 7,  7,  7,  7,  7,  7,  7,  7,  7,  7],
        [ 8,  8,  8,  8,  8,  8,  8,  8,  8,  8],
        [ 9,  9,  9,  9,  9,  9,  9,  9,  9,  9],
        [10, 10, 10, 10, 10, 10, 10, 10, 10, 10],
        [11, 11, 11, 11, 11, 11, 11, 11, 11, 11],
        [12, 12, 12, 12, 12, 12, 12, 12, 12, 12],
        [13, 13, 13, 13, 13, 13, 13, 13, 13, 13],
        [14, 14, 14, 14, 14, 14, 14, 14, 14, 14],
        [15, 15, 15, 15, 15, 15, 15, 15, 15, 15],
        [16, 16, 16, 16, 16, 16, 16, 16, 16, 16],
        [17, 17, 17, 17, 17, 17, 17, 17, 17, 17],
        [18, 18, 18, 18, 18, 18, 18, 18, 18, 18],
        [19, 19, 19, 19, 19, 19, 19, 19, 19, 19]])

#### Склейка массивов

In [ ]:
a = torch.arange(10).reshape((2, 5))
a

tensor([[0, 1, 2, 3, 4],
        [5, 6, 7, 8, 9]])

In [ ]:
b = torch.arange(20).reshape((4, 5))
b

tensor([[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19]])

In [ ]:
c = torch.arange(12).reshape((2, 6))
c

tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])

In [ ]:
torch.concat([a, a, a, a])

tensor([[0, 1, 2, 3, 4],
        [5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4],
        [5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4],
        [5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4],
        [5, 6, 7, 8, 9]])

In [ ]:
torch.concat([a, b])

tensor([[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19]])

In [ ]:
torch.concat([a, c])

RuntimeError: ignored

In [ ]:
torch.concat([a, c], axis=1)

tensor([[ 0,  1,  2,  3,  4,  0,  1,  2,  3,  4,  5],
        [ 5,  6,  7,  8,  9,  6,  7,  8,  9, 10, 11]])

In [ ]:
torch.stack([a, a, a])

tensor([[[0, 1, 2, 3, 4],
         [5, 6, 7, 8, 9]],

        [[0, 1, 2, 3, 4],
         [5, 6, 7, 8, 9]],

        [[0, 1, 2, 3, 4],
         [5, 6, 7, 8, 9]]])

In [ ]:
torch.stack([a, b])

RuntimeError: ignored

## Подборка задач для самостоятельного решения

#### Задача 1

Создайте пустой тензор размера `(10,)`.

In [ ]:
# your code here

#### Задача 2

Создайте тензор со значениями от 15 до 59.

In [ ]:
# your code here

#### Задача 3

Создайте тензор 5x5 со значениями от 100 до 124.

In [ ]:
# your code here

#### Задача 4

Создайте единичную матрицу (тензор) размера 10.

In [ ]:
# your code here

#### Задача 5

Создайте матрицу 10x20 со случайными значениями из нормального распределения. Найдите минимум:

1. Глобально,
2. По строкам,
3. По столбцам.

In [ ]:
# your code here

#### Задача 6

Создайте тензор размера 32x32 из единиц, и заполните его края нулями.

In [ ]:
# your code here

#### Задача 7

Сгенерируйте случайный тензор размера 30x30. Добавьте к нему поля из нулей (размер станет 32x32).

In [ ]:
# your code here

#### Задача 8

Создайте тензор 8x8 и заполните его 0 и 1 в шахматном порядке.

In [ ]:
# your code here

#### Задача 9

Создайте случайный тензор и отсортируйте его.

In [ ]:
# your code here

#### Задача 10

Сгенерируйте тензор из 100 случайных чисел из стандартного нормального распределения. Найдите в нем ближайшее число к 2.

In [ ]:
# your code here

#### Задача 11

Сгенерируйте 2 тензора размера `(30,)` - координаты 30 точек в двумерном пространстве. Найдите матрицу попарных расстояний для этих точек.

In [ ]:
# your code here

#### Задача 12

Создайте случайный двумерный тензор произвольного размера. Вычтите из каждой строки ее среднее.

In [ ]:
# your code here

#### Задача 13

Создайте одномерный тензор длины 100, числа в котором формируют "пилу" периода 10.

Пример "пилы" периода 3: `[1, 2, 3, 1, 2, 3, 1, 2, 3, ...]`

In [ ]:
# your code here

#### Задача 14

Создайте "пилу" длины 99 с периодом 10. Сделайте из нее тензор размера 11x9 и отсортируйте его по 2 колонке.

In [ ]:
# your code here

#### Задача 15

Посчитайте средние четырехмерного массива сразу по двум последним измерениям.

In [ ]:
# your code here

#### Задача 16*

Сгенерируйте тензор произвольной длины (более 100) и посчитайте средние в нем по окну размера 5.

In [ ]:
# your code here

## Автоматический подсчет производных

В PyTorch реализован автоматический подсчет градиентов (`torch.autograd`).

Для того, чтобы осуществлять этот расчет, PyTorch хранит **граф вычислений**, по которому возможно вычислять производную любой дифференцируемой функции, полученной из комбинации дифференцируемых функций.

![calculation_graph](https://data.bioml.ru/htdocs/courses/bioml/neural_networks/pytorch/img/calc_graph.png)

Граф вычислений для $f(x, y) = x^2 + xy + (x+y)^2$

Почему это можно посчитать по графу? Все благодаря chain rule: $$(f \circ g)'(x) = (f(g(x)))' = f'(g(x)) \cdot g'(x)$$

Посмотрим на примере.

Для того, чтобы заставить PyTorch сохранять граф вычислений, необходимо прописать параметр `requires_grad=True`.

In [ ]:
a = torch.tensor([-0.3, 0.1, 0.2], requires_grad=True)

In [ ]:
m = a.mean()
m

tensor(-4.9671e-09, grad_fn=<MeanBackward0>)

$m(a) = \dfrac{a_0 + a_1 + a_2}{3}$

In [ ]:
m.backward()

$\dfrac{\partial m}{\partial a_i} = \dfrac{1}{3}$

In [ ]:
a.grad

tensor([0.3333, 0.3333, 0.3333])

In [ ]:
a = torch.tensor([-0.3, 0.1, 0.2], requires_grad=True)

$s(a) = \dfrac{a_0^2 + a_1^2 + a_2^2}{3}$

In [ ]:
s = (a ** 2).mean()

In [ ]:
s

tensor(0.0467, grad_fn=<MeanBackward0>)

In [ ]:
s.backward()

In [ ]:
a.grad

tensor([-0.2000,  0.0667,  0.1333])

$\dfrac{\partial s}{\partial a_i} = \dfrac{2}{3}a_i$

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(-2.0, requires_grad=True)

In [ ]:
g = x + y
f = g**2 + x

In [ ]:
f.backward()

In [ ]:
x.grad, y.grad

### Ручная оптимизация простейшей модели

Рассмотрим простейшую линейную модель, работающую на 2 признаках.

In [ ]:
weights = torch.tensor([-0.1, 0.1])
weights

tensor([-0.1000,  0.1000])

Сгенерируем данные с зависимостью $y = x_1 - 2.5x_2$

In [ ]:
N = 100  # number of objects
x_1 = torch.rand(size=(N,))
x_2 = torch.randn(size=(N,))
y = x_1 - 2.5 * x_2 + torch.normal(0, 0.2, size=(N,))
objects = torch.stack([x_1, x_2]).T
objects

tensor([[ 0.8265,  0.8013],
        [ 0.8546, -1.0868],
        [ 0.3448, -0.5334],
        [ 0.7665, -0.9688],
        [ 0.3166,  0.2024],
        [ 0.1374,  0.7431],
        [ 0.1498,  0.1099],
        [ 0.1828, -1.6826],
        [ 0.1970,  1.1353],
        [ 0.7218,  1.5124],
        [ 0.1614,  1.3254],
        [ 0.1068,  0.5889],
        [ 0.8192, -1.4590],
        [ 0.7201, -0.2535],
        [ 0.6457, -0.0526],
        [ 0.8873, -1.1231],
        [ 0.8507, -0.3565],
        [ 0.5676,  0.8974],
        [ 0.3865,  0.0340],
        [ 0.4413, -0.6889],
        [ 0.6075, -1.0097],
        [ 0.9336, -1.1882],
        [ 0.4195,  0.0607],
        [ 0.7411, -1.2695],
        [ 0.3852, -0.5902],
        [ 0.3174, -0.6419],
        [ 0.1269,  0.5420],
        [ 0.3384,  0.8193],
        [ 0.0894, -1.1506],
        [ 0.8245,  0.9535],
        [ 0.1193, -1.0474],
        [ 0.0748, -2.1391],
        [ 0.7531,  0.5546],
        [ 0.9176, -2.1203],
        [ 0.9163, -0.8503],
        [ 0.8190, -0

Как получить предсказание весов?

In [ ]:
predicted = (objects * weights).sum(axis=1)
predicted

tensor([-0.0025, -0.1941, -0.0878, -0.1735, -0.0114,  0.0606, -0.0040, -0.1865,
         0.0938,  0.0791,  0.1164,  0.0482, -0.2278, -0.0974, -0.0698, -0.2010,
        -0.1207,  0.0330, -0.0352, -0.1130, -0.1617, -0.2122, -0.0359, -0.2011,
        -0.0975, -0.0959,  0.0415,  0.0481, -0.1240,  0.0129, -0.1167, -0.2214,
        -0.0199, -0.3038, -0.1767, -0.1012, -0.0812, -0.1685, -0.1654, -0.0520,
        -0.2461, -0.0663, -0.2611, -0.0971, -0.0171,  0.0824,  0.0175, -0.0103,
        -0.0200,  0.0864,  0.0774, -0.1009, -0.0360,  0.0889, -0.0845, -0.1403,
         0.1292, -0.2111, -0.1559, -0.1532, -0.0628, -0.0004,  0.0103, -0.0686,
        -0.1094, -0.0939, -0.0482, -0.1647, -0.0023, -0.1180, -0.0670,  0.0259,
         0.0396, -0.0369, -0.0701, -0.2091, -0.1010, -0.2142, -0.0014, -0.1039,
         0.0497, -0.0379, -0.1591,  0.0103, -0.0358,  0.0344,  0.0622, -0.1502,
         0.0402, -0.2304, -0.0395,  0.1263, -0.0911, -0.1485,  0.0967, -0.1034,
        -0.3361,  0.0585,  0.0674,  0.02

Как понять, насколько сильно мы ошиблись?

In [ ]:
mse = ((predicted - y) ** 2).mean()
mse

tensor(7.6811)

In [ ]:
mse.backward()

RuntimeError: ignored

Что нужно исправить?

In [ ]:
weights = torch.tensor([-0.1, 0.1], requires_grad=True)
weights

tensor([-0.1000,  0.1000], requires_grad=True)

In [ ]:
predicted = (objects * weights).sum(axis=1)
mse = ((predicted - y) ** 2).mean()
mse

tensor(0.0393, grad_fn=<MeanBackward0>)

In [ ]:
mse.backward()

Смотрим на градиент:

In [ ]:
weights

tensor([ 1.0112, -2.5083], requires_grad=True)

In [ ]:
weights.grad

tensor([-0.0304,  0.0004])

Что нужно сделать с градиентом, чтобы исправить веса?

In [ ]:
weights.data = weights - weights.grad * 0.1

In [ ]:
weights.grad.data.zero_()
#weights.requires_grad_(True)

tensor([0., 0.])

#### Задача

Автоматизируйте оптимизацию градиента самостоятельно.

In [ ]:
# your code here

### Другие операции с градиентом

Часть операций бывает нужно выполнить без сохранения градиента. Например, такими операциями будут промежуточные оценки качества модели в процессе обучения.

In [ ]:
a = torch.tensor([1.0, 2.0], requires_grad=True)

with torch.no_grad():
    b = a.sum()

Как мы видим, граф не сохранялся:

In [ ]:
b.backward()

RuntimeError: ignored

Будьте внимательны: каждая операция с тензором, для которого требуется подсчет градиентов, активирует режим подсчета градиентов для тензоров на выходе. В результате при неаккуратном использовании видеопамять может оказаться полностью забитой графом вычислений.

Тензоры весов модели имеют параметр `requires_grad=True` по умолчанию!